# 태양풍 속도 예측 — P1 적용본

베이스라인 대비 변경점 (P1):

1. **Persistence 잔차 예측** — `target` 대신 `target - wind_19`를 학습하고 추론 시 복원
2. **Wind branch 강화** — 원시값 20개 MLP → `[값, 1차차분, 결측플래그]` 시퀀스 GRU + 통계 피처
3. **정규화 개선** — train split 통계로 이미지/윈드/잔차 표준화 (per-image 표준화는 코로나홀 정보를 파괴하므로 사용하지 않음)
4. **손실함수를 공식 지표에 정렬** — `mean_h( RMSE_h )` 를 직접 최소화
5. **해상도 64 → 128** (스템에서 두 단계 다운샘플링해 메모리/연산은 기존 수준 유지)

추가: 공식 평가식(horizon별 RMSE의 평균) 구현, 윈드 결측 처리, 예측값 물리 범위 클리핑.

---

### P2a — 과적합 억제 (1차 실험 결과 반영)

1차 실행에서 **best epoch 2**, train 37 / val 73 으로 벌어졌습니다. 샘플이 6h stride 슬라이딩 윈도우라
연속 샘플이 19/20을 공유하고, **1 epoch ≈ 독립 데이터 20회 통과**이기 때문입니다.

6. **증강** — 밝기/대비 지터, 평행이동, 노이즈, cutout. **좌우 반전은 사용하지 않음** (자전 방향이 뒤집힘)
7. **용량 축소** — `flatten(2048)` → (위도, 경도) 격자 평균 풀링 `(1,4)` = 512. 경도는 도달 시간을 결정하므로 보존
8. **정규화 강화** — dropout 0.2 → 0.4, weight decay 1e-4 → 1e-3
9. **`USE_IMAGES` 스위치** — `False` 로 두면 윈드 전용 모델. 영상 브랜치의 실제 기여도를 측정하는 ablation용

> ⚠️ Validation은 **모델 선택/조기종료 용도로만** 사용합니다. 학습 데이터에 절대 포함하지 않습니다.
> ⚠️ Pretrained weight를 불러오지 않고 전부 random init에서 학습합니다.

## 0. 설정

In [ ]:
from pathlib import Path
import gc
import json
import math
import os
import random
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 777
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 데이터 경로 자동 탐색: 플랫폼마다 위치가 다를 수 있어 후보를 순회합니다.
DATA_ROOT_CANDIDATES = [
    Path(os.getenv("SW_DATA_ROOT", "")) if os.getenv("SW_DATA_ROOT") else None,
    Path("public_dataset/competition_dataset_6h"),
    Path("/home/jovyan/public_dataset/competition_dataset_6h"),
    Path("public/public_dataset/competition_dataset_6h"),
    Path("/home/jovyan/public/public_dataset/competition_dataset_6h"),
    Path("dataset"),
    Path("/home/jovyan/dataset"),
]
DATA_ROOT = None
for candidate in DATA_ROOT_CANDIDATES:
    if candidate is not None and (candidate / "train/inputs.csv").exists():
        DATA_ROOT = candidate
        break
if DATA_ROOT is None:
    searched = "\n".join(f"  - {c}" for c in DATA_ROOT_CANDIDATES if c is not None)
    raise FileNotFoundError(
        "데이터 경로를 찾지 못했습니다. 아래 후보를 확인했습니다:\n" + searched
        + "\n\nSW_DATA_ROOT 환경변수나 DATA_ROOT를 직접 지정하세요."
    )

# 이미지 cache 는 dataset 폴더가 read-only 일 수 있으므로 작업 폴더 아래에 만듭니다.
WORK_DIR = Path("work")
CACHE_ROOT = WORK_DIR / "cache"
OUTPUT_DIR = WORK_DIR / "outputs"
SUBMISSION_DIR = Path("submission")
for directory in (CACHE_ROOT, OUTPUT_DIR, SUBMISSION_DIR):
    directory.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 128          # P1-5: 64 -> 128
CHANNELS = ("193", "211")
BATCH_SIZE = 64           # 128px 는 활성값이 커집니다. OOM 이면 48 / 32 로 낮추세요.
EPOCHS = 60
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-3       # P2a: 1e-4 -> 1e-3
GRAD_CLIP = 1.0
SCHEDULER_PATIENCE = 3
EARLY_STOP_PATIENCE = 10
NUM_WORKERS = 4

# --- P2a: 과적합 억제 -------------------------------------------------
# best epoch 2 에서 조기종료되고 train 37 / val 73 으로 벌어진 문제 대응.
# 샘플이 6h stride 슬라이딩 윈도우라 연속 샘플이 19/20 을 공유합니다.
# 즉 1 epoch = 독립 데이터 약 20회 통과. 강한 정규화가 필요합니다.
USE_IMAGES = True         # False 로 두면 윈드 전용 모델 (영상 기여도 측정용 ablation)
DROPOUT = 0.4             # P2a: 0.2 -> 0.4
POOL_GRID = (1, 4)        # (위도, 경도) 풀링 격자. 위도는 뭉개고 경도 4구간은 보존
AUGMENT = True
AUG_BRIGHTNESS = 0.10     # 밝기/대비 지터 폭
AUG_SHIFT_PIXELS = 6      # 전 프레임 공통 평행이동 (관측 포인팅 지터 모사)
AUG_NOISE_STD = 0.02
AUG_ERASE_PROB = 0.3      # cutout 확률
LOSS_EPSILON = 1e-8
LOSS_SCALE = 100.0        # km/s -> O(1) 로 맞춰 gradient 크기를 안정화
LOSS_MODE = "metric"      # "metric" = mean_h RMSE_h (공식 지표), "mse" = 일반 MSE

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
PIN_MEMORY = DEVICE.type == "cuda"
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
else:
    print("WARNING: CUDA Unavailable")

print("PyTorch:", torch.__version__)
print("device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("data:", DATA_ROOT.resolve())

## 1. 데이터 로드 · 무결성 검사 · 결측 처리

베이스라인의 검사에 더해 **wind / target 의 결측(NaN)** 을 확인합니다.
L1 관측 기반 태양풍 데이터는 실제로 결손 구간이 흔하고, 베이스라인은 이를 전혀 처리하지 않습니다.

In [ ]:
IMAGE_COLUMNS = [f"image_{index:02d}" for index in range(20)]
WIND_COLUMNS = [f"wind_{index:02d}" for index in range(20)]
TARGET_COLUMNS = [f"target_{index:02d}" for index in range(12)]
HORIZONS = np.arange(1, 13) * 6

train_inputs = pd.read_csv(DATA_ROOT / "train/inputs.csv")
train_targets_frame = pd.read_csv(DATA_ROOT / "train/targets.csv")
val_inputs = pd.read_csv(DATA_ROOT / "validation/inputs.csv")
val_targets_frame = pd.read_csv(DATA_ROOT / "validation/targets.csv")
test_inputs = pd.read_csv(DATA_ROOT / "test/inputs.csv")
test_ids = pd.read_csv(DATA_ROOT / "test/test_ids.csv")

assert train_inputs.sample_id.tolist() == train_targets_frame.sample_id.tolist()
assert val_inputs.sample_id.tolist() == val_targets_frame.sample_id.tolist()
assert test_inputs.sample_id.tolist() == test_ids.sample_id.tolist()
assert train_inputs.sample_id.is_unique and val_inputs.sample_id.is_unique
assert test_inputs.sample_id.is_unique
assert set(train_inputs.sample_id).isdisjoint(val_inputs.sample_id)
assert set(train_inputs.sample_id).isdisjoint(test_inputs.sample_id)
assert set(val_inputs.sample_id).isdisjoint(test_inputs.sample_id)
assert not any(column.startswith("target_") for column in test_inputs.columns)


def forward_fill_rows(values):
    # 행 방향(시간축) forward fill. 유효값이 하나도 없는 행은 그대로 NaN 으로 남습니다.
    valid = np.isfinite(values)
    positions = np.where(valid, np.arange(values.shape[1])[None, :], 0)
    np.maximum.accumulate(positions, axis=1, out=positions)
    rows = np.arange(values.shape[0])[:, None]
    return np.where(valid.any(axis=1, keepdims=True), values[rows, positions], values)


def fill_wind(frame, fallback):
    # forward fill -> backward fill -> 전역 대푯값 순으로 메웁니다.
    values = frame[WIND_COLUMNS].to_numpy(np.float32)
    valid = np.isfinite(values).astype(np.float32)
    filled = forward_fill_rows(values)
    filled = forward_fill_rows(filled[:, ::-1])[:, ::-1]
    filled = np.where(np.isfinite(filled), filled, fallback)
    return np.ascontiguousarray(filled), np.ascontiguousarray(valid)


train_wind_raw = train_inputs[WIND_COLUMNS].to_numpy(np.float32)
WIND_FALLBACK = float(np.nanmedian(train_wind_raw))

train_wind, train_wind_valid = fill_wind(train_inputs, WIND_FALLBACK)
val_wind, val_wind_valid = fill_wind(val_inputs, WIND_FALLBACK)
test_wind, test_wind_valid = fill_wind(test_inputs, WIND_FALLBACK)

train_targets = train_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)
val_targets = val_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)

print("=== 결측 점검 ===")
for name, frame, columns in [
    ("train wind", train_inputs, WIND_COLUMNS),
    ("val wind", val_inputs, WIND_COLUMNS),
    ("test wind", test_inputs, WIND_COLUMNS),
    ("train target", train_targets_frame, TARGET_COLUMNS),
    ("val target", val_targets_frame, TARGET_COLUMNS),
]:
    array = frame[columns].to_numpy(np.float32)
    missing = int((~np.isfinite(array)).sum())
    print(f"{name:14s} NaN {missing:>7,} / {array.size:>9,}  ({missing / array.size:.4%})")

assert np.isfinite(train_targets).all(), "train target 에 NaN 이 있습니다. 처리 방침을 정하세요."
assert np.isfinite(val_targets).all(), "val target 에 NaN 이 있습니다."

print()
print("=== 분포 ===")
print(f"train wind   mean={train_wind.mean():7.2f}  std={train_wind.std():6.2f}"
      f"  min={train_wind.min():6.1f}  max={train_wind.max():6.1f}")
print(f"train target mean={train_targets.mean():7.2f}  std={train_targets.std():6.2f}"
      f"  min={train_targets.min():6.1f}  max={train_targets.max():6.1f}")
print("samples:", len(train_inputs), len(val_inputs), len(test_inputs))

## 2. 128px 이미지 memory-map cache

슬라이딩 윈도우라 같은 PNG 가 여러 샘플에 반복 등장합니다. 고유 파일당 한 번만 resize 해서 저장합니다.
128px 기준 총 용량은 약 500MB 수준입니다 (64px 의 4배).

> VM 디스크를 과도하게 쓰면 인스턴스가 내려간 사례가 있습니다. 해상도를 더 올릴 때는 용량을 먼저 계산하세요.

In [ ]:
def prepare_image_memmap(split, inputs):
    image_root = DATA_ROOT / split
    cache_root = CACHE_ROOT / f"{IMAGE_SIZE}px"
    cache_root.mkdir(parents=True, exist_ok=True)
    array_path = cache_root / f"{split}_images.npy"
    metadata_path = cache_root / f"{split}_metadata.json"

    filenames = sorted(pd.unique(inputs[IMAGE_COLUMNS].to_numpy().ravel()).tolist())
    expected = {
        "image_size": IMAGE_SIZE,
        "channels": list(CHANNELS),
        "filenames": filenames,
    }
    shape = (len(filenames), len(CHANNELS), IMAGE_SIZE, IMAGE_SIZE)

    valid = False
    if array_path.exists() and metadata_path.exists():
        try:
            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            cached = np.load(array_path, mmap_mode="r")
            valid = (metadata == expected and cached.shape == shape
                     and cached.dtype == np.uint8)
        except (OSError, ValueError, json.JSONDecodeError):
            valid = False

    if not valid:
        array_temp = array_path.with_name(array_path.name + f".partial.{os.getpid()}")
        metadata_temp = metadata_path.with_name(metadata_path.name + f".partial.{os.getpid()}")
        resized = np.lib.format.open_memmap(
            array_temp, mode="w+", dtype=np.uint8, shape=shape
        )
        resampling = Image.Resampling.BILINEAR
        for index, filename in enumerate(filenames):
            for channel_index, channel in enumerate(CHANNELS):
                with Image.open(image_root / channel / filename) as image:
                    resized[index, channel_index] = np.asarray(
                        image.convert("L").resize((IMAGE_SIZE, IMAGE_SIZE), resampling),
                        dtype=np.uint8,
                    )
            if (index + 1) % 2000 == 0 or index + 1 == len(filenames):
                print(f"{split} resize: {index + 1}/{len(filenames)}", flush=True)
        resized.flush()
        del resized
        metadata_temp.write_text(json.dumps(expected, ensure_ascii=False) + "\n", encoding="utf-8")
        array_temp.replace(array_path)
        metadata_temp.replace(metadata_path)
        print(f"created cache: {array_path.resolve()}")
    else:
        print(f"reusing cache: {array_path.resolve()}")

    image_array = np.load(array_path, mmap_mode="r")
    image_index = {filename: index for index, filename in enumerate(filenames)}
    return image_array, image_index


train_image_array, train_image_index = prepare_image_memmap("train", train_inputs)
val_image_array, val_image_index = prepare_image_memmap("validation", val_inputs)
test_image_array, test_image_index = prepare_image_memmap("test", test_inputs)

total_bytes = sum(a.nbytes for a in (train_image_array, val_image_array, test_image_array))
print(f"cache 총 용량: {total_bytes / 1024 ** 3:.2f} GiB")
print("고유 이미지 수:", len(train_image_index), len(val_image_index), len(test_image_index))

## 3. 정규화 통계 — **train split 에서만** 계산

- **이미지**: 채널별 전역 mean/std. per-image 표준화를 쓰지 않는 이유는, 코로나홀이 "절대적으로 어두운" 영역이기 때문입니다. 이미지마다 밝기를 맞추면 고속 태양풍의 핵심 신호가 사라집니다.
- **윈드**: 전역 mean/std, 1차 차분은 별도 std.
- **잔차**: `target_h - wind_19` 의 horizon별 mean/std. 모델 출력단 affine 으로 넣어 초기 스케일을 맞춥니다.

In [ ]:
def compute_image_stats(array, chunk=256):
    total = np.zeros(len(CHANNELS), np.float64)
    total_square = np.zeros(len(CHANNELS), np.float64)
    count = 0
    for start in range(0, len(array), chunk):
        block = np.asarray(array[start:start + chunk], dtype=np.float64) / 255.0
        total += block.sum(axis=(0, 2, 3))
        total_square += (block ** 2).sum(axis=(0, 2, 3))
        count += block.shape[0] * block.shape[2] * block.shape[3]
    mean = total / count
    variance = np.maximum(total_square / count - mean ** 2, 1e-12)
    return mean.astype(np.float32), np.sqrt(variance).astype(np.float32)


IMAGE_MEAN, IMAGE_STD = compute_image_stats(train_image_array)

WIND_MEAN = float(train_wind.mean())
WIND_STD = float(train_wind.std() + 1e-6)
train_wind_diff = np.diff(train_wind, axis=1, prepend=train_wind[:, :1])
DIFF_STD = float(train_wind_diff.std() + 1e-6)

# P1-1: persistence 잔차. 마지막 관측값 대비 변화량만 학습합니다.
train_residual = train_targets - train_wind[:, -1:]
RESIDUAL_MEAN = train_residual.mean(axis=0).astype(np.float32)
RESIDUAL_STD = (train_residual.std(axis=0) + 1e-6).astype(np.float32)

# 예측값 클리핑 범위 (train target 실측 범위에 여유를 둔 물리적 경계)
CLIP_LOW = float(train_targets.min() * 0.95)
CLIP_HIGH = float(train_targets.max() * 1.05)

print("image mean:", IMAGE_MEAN, " std:", IMAGE_STD)
print(f"wind mean={WIND_MEAN:.2f} std={WIND_STD:.2f} diff_std={DIFF_STD:.2f}")
print("residual mean by horizon:", np.round(RESIDUAL_MEAN, 1))
print("residual std  by horizon:", np.round(RESIDUAL_STD, 1))
print(f"clip range: [{CLIP_LOW:.1f}, {CLIP_HIGH:.1f}] km/s")

## 4. Dataset / DataLoader

**P1-2**: wind branch 입력을 두 갈래로 만듭니다.

- `wind_seq` `(20, 3)` = `[표준화 값, 표준화 1차차분, 결측 플래그]` → GRU
- `wind_stats` `(9,)` = 마지막값 / 최근 4개 평균 / 전체 평균·표준편차 / min / max / 선형기울기 / (마지막 − 최근4평균) / (max − min)

In [ ]:
STAT_NAMES = ["last", "mean4", "mean", "std", "min", "max", "slope", "last_minus_mean4", "range"]
_TIME_AXIS = np.arange(20, dtype=np.float32)
_TIME_CENTERED = _TIME_AXIS - _TIME_AXIS.mean()
_TIME_DENOMINATOR = float((_TIME_CENTERED ** 2).sum())


def build_wind_stats(wind):
    last = wind[:, -1]
    mean4 = wind[:, -4:].mean(axis=1)
    slope = (wind - wind.mean(axis=1, keepdims=True)) @ _TIME_CENTERED / _TIME_DENOMINATOR
    stats = np.stack([
        last,
        mean4,
        wind.mean(axis=1),
        wind.std(axis=1),
        wind.min(axis=1),
        wind.max(axis=1),
        slope,
        last - mean4,
        wind.max(axis=1) - wind.min(axis=1),
    ], axis=1)
    return stats.astype(np.float32)


train_stats_raw = build_wind_stats(train_wind)
STATS_MEAN = train_stats_raw.mean(axis=0).astype(np.float32)
STATS_STD = (train_stats_raw.std(axis=0) + 1e-6).astype(np.float32)
NUM_STATS = len(STAT_NAMES)
print(pd.DataFrame({"stat": STAT_NAMES, "mean": STATS_MEAN, "std": STATS_STD}))


class SolarWindDataset(Dataset):
    def __init__(self, image_array, image_index, inputs, wind, wind_valid,
                 targets=None, training=False):
        self.training = training
        self.image_array = image_array
        self.image_indexes = np.asarray([
            [image_index[filename] for filename in row]
            for row in inputs[IMAGE_COLUMNS].itertuples(index=False, name=None)
        ], dtype=np.int32)
        self.sample_ids = inputs.sample_id.to_numpy()

        self.last_wind = np.ascontiguousarray(wind[:, -1]).astype(np.float32)
        wind_normalized = (wind - WIND_MEAN) / WIND_STD
        wind_diff = np.diff(wind, axis=1, prepend=wind[:, :1]) / DIFF_STD
        self.wind_seq = np.stack(
            [wind_normalized, wind_diff, wind_valid], axis=2
        ).astype(np.float32)
        self.wind_stats = ((build_wind_stats(wind) - STATS_MEAN) / STATS_STD).astype(np.float32)

        self.targets = targets.astype(np.float32) if targets is not None else None
        self.image_mean = IMAGE_MEAN.reshape(1, len(CHANNELS), 1, 1)
        self.image_std = IMAGE_STD.reshape(1, len(CHANNELS), 1, 1)

    def __len__(self):
        return len(self.sample_ids)

    def augment(self, images):
        # 좌우 반전은 쓰지 않습니다: 태양 자전 방향(동->서)이 뒤집혀 물리가 깨집니다.
        if AUG_SHIFT_PIXELS > 0:
            shift_y = np.random.randint(-AUG_SHIFT_PIXELS, AUG_SHIFT_PIXELS + 1)
            shift_x = np.random.randint(-AUG_SHIFT_PIXELS, AUG_SHIFT_PIXELS + 1)
            # 20 프레임에 같은 이동을 적용해야 시간축 일관성이 유지됩니다.
            images = np.roll(images, (shift_y, shift_x), axis=(2, 3))
        scale = 1.0 + np.random.uniform(-AUG_BRIGHTNESS, AUG_BRIGHTNESS)
        offset = np.random.normal(0.0, AUG_BRIGHTNESS)
        images = images * scale + offset
        if AUG_NOISE_STD > 0:
            images = images + np.random.normal(
                0.0, AUG_NOISE_STD, images.shape
            ).astype(np.float32)
        if np.random.random() < AUG_ERASE_PROB:
            size = max(4, IMAGE_SIZE // 8)
            top = np.random.randint(0, IMAGE_SIZE - size)
            left = np.random.randint(0, IMAGE_SIZE - size)
            images[:, :, top:top + size, left:left + size] = 0.0
        return images

    def __getitem__(self, item):
        if USE_IMAGES:
            images = np.asarray(
                self.image_array[self.image_indexes[item]], dtype=np.float32
            ) / 255.0
            images = (images - self.image_mean) / self.image_std
            if self.training and AUGMENT:
                images = self.augment(images)
            images = images.astype(np.float32)
        else:
            # ablation: 영상을 아예 읽지 않아 학습이 훨씬 빨라집니다.
            images = np.zeros((1, 1, 1, 1), dtype=np.float32)
        result = {
            "images": torch.from_numpy(np.ascontiguousarray(images)),
            "wind_seq": torch.from_numpy(self.wind_seq[item]),
            "wind_stats": torch.from_numpy(self.wind_stats[item]),
            "last_wind": torch.tensor(self.last_wind[item]),
            "sample_id": self.sample_ids[item],
        }
        if self.targets is not None:
            result["target"] = torch.from_numpy(self.targets[item])
        return result


def seed_worker(worker_id):
    worker_seed = (SEED + worker_id) % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def make_loader(dataset, shuffle):
    options = dict(
        dataset=dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(SEED),
    )
    if NUM_WORKERS > 0:
        options.update(persistent_workers=True, prefetch_factor=2)
    return DataLoader(**options)


train_dataset = SolarWindDataset(
    train_image_array, train_image_index, train_inputs,
    train_wind, train_wind_valid, train_targets, training=True,
)
val_dataset = SolarWindDataset(
    val_image_array, val_image_index, val_inputs,
    val_wind, val_wind_valid, val_targets,
)
train_loader = make_loader(train_dataset, shuffle=True)
val_loader = make_loader(val_dataset, shuffle=False)

batch = next(iter(train_loader))
if USE_IMAGES:
    assert batch["images"].shape[1:] == (20, len(CHANNELS), IMAGE_SIZE, IMAGE_SIZE)
assert batch["wind_seq"].shape[1:] == (20, 3)
assert batch["wind_stats"].shape[1:] == (NUM_STATS,)
assert batch["target"].shape[1:] == (12,)
print({key: tuple(value.shape) for key, value in batch.items() if torch.is_tensor(value)})

## 5. 모델

**P1-5** 를 수용하기 위한 최소 변경만 했습니다. 128px 입력을 스템에서 두 번 다운샘플(→32×32)한 뒤,
베이스라인과 동일하게 Inception3D 3단을 거쳐 4×4 로 만듭니다. LSTM 입력 차원은 2048 로 동일합니다.

**P1-2** wind branch: GRU(3→96, 2층) + 통계 MLP.
**P1-1** 출력: 표준화된 잔차 → `residual_std * z + residual_mean` (buffer 로 저장되어 추론 시 자동 복원).

> 영상 인코더 자체의 교체(프레임별 2D CNN + attention pooling)는 P2 작업입니다.

In [ ]:
class Inception3D(nn.Module):
    def __init__(self, in_channels, branch_channels=32):
        super().__init__()
        self.branch_1 = nn.Sequential(
            nn.Conv3d(in_channels, branch_channels, 1), nn.ReLU(inplace=True)
        )
        self.branch_3 = nn.Sequential(
            nn.Conv3d(in_channels, branch_channels, 1), nn.ReLU(inplace=True),
            nn.Conv3d(branch_channels, branch_channels, (1, 3, 3), padding=(0, 1, 1)),
            nn.ReLU(inplace=True),
        )
        self.branch_5 = nn.Sequential(
            nn.Conv3d(in_channels, branch_channels, 1), nn.ReLU(inplace=True),
            nn.Conv3d(branch_channels, branch_channels, (1, 5, 5), padding=(0, 2, 2)),
            nn.ReLU(inplace=True),
        )
        self.branch_pool = nn.Sequential(
            nn.MaxPool3d((1, 3, 3), stride=1, padding=(0, 1, 1)),
            nn.Conv3d(in_channels, branch_channels, 1), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return torch.cat(
            [self.branch_1(x), self.branch_3(x), self.branch_5(x), self.branch_pool(x)],
            dim=1,
        )


class SolarWindP1(nn.Module):
    def __init__(self, residual_mean, residual_std, num_stats):
        super().__init__()
        # 128 -> 64 -> 32 : 비싼 Inception 을 고해상도에서 돌리지 않도록 스템에서 줄입니다.
        self.stem = nn.Sequential(
            nn.Conv3d(len(CHANNELS), 32, (1, 5, 5), padding=(0, 2, 2)),
            nn.BatchNorm3d(32), nn.ReLU(inplace=True),
            nn.MaxPool3d((1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1)),
            nn.Conv3d(32, 64, (1, 3, 3), padding=(0, 1, 1)),
            nn.BatchNorm3d(64), nn.ReLU(inplace=True),
            nn.MaxPool3d((1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1)),
        )
        blocks = []
        in_channels = 64
        for _ in range(3):
            blocks.extend([
                Inception3D(in_channels, 32),
                nn.MaxPool3d((1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1)),
            ])
            in_channels = 128
        self.image_encoder = nn.Sequential(*blocks)

        # P2a: flatten 대신 (위도, 경도) 격자 평균 풀링.
        # 위도는 뭉개도 되지만 경도는 자전으로 지구 도달 시간을 결정하므로 보존합니다.
        # 128*4*4=2048 -> 128*1*4=512 로 LSTM 파라미터가 4배 줄어듭니다.
        was_training = self.training
        self.eval()
        with torch.no_grad():
            probe = self.image_encoder(
                self.stem(torch.zeros(1, len(CHANNELS), 2, IMAGE_SIZE, IMAGE_SIZE))
            )
        self.train(was_training)
        frame_features = probe.shape[1] * POOL_GRID[0] * POOL_GRID[1]
        image_dim = 128 if USE_IMAGES else 0
        if USE_IMAGES:
            print(f"image encoder: {IMAGE_SIZE}px -> {probe.shape[3]}x{probe.shape[4]}"
                  f" x {probe.shape[1]}ch -> pool{POOL_GRID} = {frame_features} per frame")
            self.image_lstm = nn.LSTM(
                input_size=frame_features, hidden_size=128, batch_first=True
            )
            self.image_dropout = nn.Dropout(DROPOUT)
        else:
            print("USE_IMAGES=False : 윈드 전용 모델 (영상 브랜치 없음)")

        self.wind_gru = nn.GRU(input_size=3, hidden_size=96, num_layers=2, batch_first=True)
        self.stats_encoder = nn.Sequential(
            nn.Linear(num_stats, 128), nn.SELU(inplace=True),
            nn.Linear(128, 64), nn.SELU(inplace=True),
        )
        self.head = nn.Sequential(
            nn.Linear(image_dim + 96 + 64, 256), nn.ReLU(inplace=True),
            nn.Dropout(DROPOUT),
            nn.Linear(256, 128), nn.ReLU(inplace=True),
            nn.Dropout(DROPOUT),
            nn.Linear(128, 12),
        )
        # 잔차 통계를 buffer 로 보관 -> state_dict 에 함께 저장되어 추론 시 자동 복원됩니다.
        self.register_buffer("residual_mean", torch.as_tensor(residual_mean, dtype=torch.float32))
        self.register_buffer("residual_std", torch.as_tensor(residual_std, dtype=torch.float32))

    def forward(self, images, wind_seq, wind_stats):
        _, wind_hidden = self.wind_gru(wind_seq)
        wind_features = F.relu(wind_hidden[-1])
        stats_features = self.stats_encoder(wind_stats)
        parts = [wind_features, stats_features]

        if USE_IMAGES:
            features = images.permute(0, 2, 1, 3, 4).contiguous()  # (B,T,C,H,W)->(B,C,T,H,W)
            features = self.image_encoder(self.stem(features))
            # 시간축은 그대로 두고 공간만 (위도, 경도) 격자로 평균 풀링
            features = F.adaptive_avg_pool3d(
                features, (features.shape[2], POOL_GRID[0], POOL_GRID[1])
            )
            features = features.permute(0, 2, 1, 3, 4).flatten(2)  # (B,T,C*gy*gx)
            _, (hidden, _) = self.image_lstm(features)
            parts.insert(0, self.image_dropout(F.relu(hidden[-1])))

        z = self.head(torch.cat(parts, dim=1))
        return z * self.residual_std + self.residual_mean   # 잔차(km/s)


def build_model():
    return SolarWindP1(RESIDUAL_MEAN, RESIDUAL_STD, NUM_STATS).to(DEVICE)


model = build_model()
print("trainable parameters:",
      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 6. 공식 평가 지표

사전 설명회 자료 p.27 정의:

$$\mathrm{RMSE}_h=\sqrt{\frac{1}{N}\sum_{i=1}^{N}(y_{h,i}-\hat y_{h,i})^2},\qquad
\mathrm{RMSE}=\frac{1}{H}\sum_{h=1}^{H}\mathrm{RMSE}_h$$

**horizon별 RMSE 를 먼저 낸 뒤 평균**입니다. 전체 원소를 한꺼번에 묶은 RMSE 와는 값이 다릅니다 (아래에서 둘 다 출력).
손실함수도 같은 형태로 맞춥니다 (**P1-4**).

In [ ]:
def official_rmse(y_true, y_pred):
    # (1/H) * sum_h RMSE_h
    per_horizon = np.sqrt(np.mean((y_pred - y_true) ** 2, axis=0))
    return float(per_horizon.mean()), per_horizon


def pooled_rmse(y_true, y_pred):
    # 전체 원소를 묶은 RMSE (베이스라인이 출력하던 값)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))


def metrics_by_horizon(y_true, y_pred, persistence=None):
    rows = []
    for index in range(12):
        actual, predicted = y_true[:, index], y_pred[:, index]
        error = predicted - actual
        denominator = np.std(actual) * np.std(predicted)
        row = {
            "horizon_h": int(HORIZONS[index]),
            "rmse": float(np.sqrt(np.mean(error ** 2))),
            "mae": float(np.mean(np.abs(error))),
            "corr": float(np.corrcoef(actual, predicted)[0, 1]) if denominator > 0 else np.nan,
        }
        if persistence is not None:
            row["persistence_rmse"] = float(
                np.sqrt(np.mean((persistence[:, index] - actual) ** 2))
            )
            row["gain"] = row["persistence_rmse"] - row["rmse"]
        rows.append(row)
    return pd.DataFrame(rows)


def metric_loss(prediction, target):
    error = (prediction - target) / LOSS_SCALE
    if LOSS_MODE == "mse":
        return F.mse_loss(error, torch.zeros_like(error))
    # batch 내 horizon별 RMSE 의 평균 -> 공식 지표와 동일한 가중
    per_horizon_mse = (error ** 2).mean(dim=0)
    return torch.sqrt(per_horizon_mse + LOSS_EPSILON).mean()


# persistence 기준선: 마지막 관측 wind 가 72시간 유지된다고 가정
val_persistence = np.repeat(val_wind[:, -1:], 12, axis=1).astype(np.float64)
persistence_score, persistence_per_horizon = official_rmse(val_targets, val_persistence)
print(f"[기준선] persistence 공식 RMSE = {persistence_score:.3f} km/s")
print("horizon별:", np.round(persistence_per_horizon, 1))
print(f"[기준선] persistence pooled RMSE = {pooled_rmse(val_targets, val_persistence):.3f} km/s")

## 7. 학습

- 매 실행마다 random init (pretrained weight 없음)
- validation 은 **early stopping / best checkpoint 선택** 에만 사용
- AdamW + gradient clipping, `ReduceLROnPlateau(factor=0.5, patience=3)`
- 손실은 AMP 밖 float32 에서 계산 (sqrt 안정성)

In [ ]:
TRAIN_FROM_SCRATCH = True   # 재현 검증 시에도 True 로 두고 처음부터 실행되게 합니다.

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model = build_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=SCHEDULER_PATIENCE, min_lr=1e-6
)
scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)

checkpoint_path = OUTPUT_DIR / "best_model.pth"
if checkpoint_path.exists():
    checkpoint_path.unlink()

best_val_score = float("inf")
epochs_without_improvement = 0
history = []


def run_epoch(loader, training):
    model.train(training)
    squared_error_sum = np.zeros(12, dtype=np.float64)
    sample_count = 0
    for batch in loader:
        images = batch["images"].to(DEVICE, non_blocking=PIN_MEMORY)
        wind_seq = batch["wind_seq"].to(DEVICE, non_blocking=PIN_MEMORY)
        wind_stats = batch["wind_stats"].to(DEVICE, non_blocking=PIN_MEMORY)
        last_wind = batch["last_wind"].to(DEVICE, non_blocking=PIN_MEMORY)
        target = batch["target"].to(DEVICE, non_blocking=PIN_MEMORY)

        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
                residual = model(images, wind_seq, wind_stats)
            # P1-1: 잔차 + 마지막 관측값 = 절대 속도. loss 는 float32 에서.
            prediction = residual.float() + last_wind.unsqueeze(1)
            loss = metric_loss(prediction, target)
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

        error = (prediction.detach() - target).double()
        squared_error_sum += torch.sum(error ** 2, dim=0).cpu().numpy()
        sample_count += error.shape[0]

    per_horizon = np.sqrt(squared_error_sum / sample_count)
    return float(per_horizon.mean()), per_horizon


for epoch in range(1, EPOCHS + 1):
    started = time.perf_counter()
    train_score, _ = run_epoch(train_loader, training=True)
    with torch.no_grad():
        val_score, val_per_horizon = run_epoch(val_loader, training=False)
    scheduler.step(val_score)
    learning_rate = optimizer.param_groups[0]["lr"]
    elapsed = time.perf_counter() - started

    history.append({
        "epoch": epoch,
        "train_rmse": train_score,
        "val_rmse": val_score,
        "learning_rate": learning_rate,
        "seconds": elapsed,
    })
    marker = ""
    if val_score < best_val_score:
        best_val_score = val_score
        epochs_without_improvement = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "val_official_rmse": val_score,
                "image_size": IMAGE_SIZE,
                "channels": list(CHANNELS),
                "image_mean": IMAGE_MEAN.tolist(),
                "image_std": IMAGE_STD.tolist(),
                "wind_mean": WIND_MEAN,
                "wind_std": WIND_STD,
                "diff_std": DIFF_STD,
                "stats_mean": STATS_MEAN.tolist(),
                "stats_std": STATS_STD.tolist(),
                "residual_mean": RESIDUAL_MEAN.tolist(),
                "residual_std": RESIDUAL_STD.tolist(),
                "wind_fallback": WIND_FALLBACK,
                "clip_low": CLIP_LOW,
                "clip_high": CLIP_HIGH,
                "seed": SEED,
                "initialization": "random_from_scratch",
            },
            checkpoint_path,
        )
        marker = "  <- best"
    else:
        epochs_without_improvement += 1

    print(f"epoch={epoch:03d} train={train_score:7.3f} val={val_score:7.3f} "
          f"lr={learning_rate:.2e} {elapsed:6.1f}s{marker}", flush=True)

    if epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print("early stopping")
        break

history_frame = pd.DataFrame(history)
history_frame.to_csv(OUTPUT_DIR / "history.csv", index=False)

figure, axis = plt.subplots(figsize=(7, 4))
axis.plot(history_frame.epoch, history_frame.train_rmse, label="train")
axis.plot(history_frame.epoch, history_frame.val_rmse, label="validation")
axis.axhline(persistence_score, color="gray", linestyle="--", label="persistence")
axis.set_xlabel("epoch")
axis.set_ylabel("official RMSE (km/s)")
axis.grid(alpha=0.3)
axis.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "learning_curve.png", dpi=140)
plt.show()

checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"best epoch {checkpoint['epoch']}  val official RMSE {checkpoint['val_official_rmse']:.3f}")

## 8. Validation 평가 · persistence 대비 확인

**persistence 를 못 이기면 학습이 의미 있게 되지 않은 것입니다.** 여기서 먼저 확인하세요.

In [ ]:
@torch.no_grad()
def predict(loader):
    model.eval()
    predictions, sample_ids = [], []
    for batch in loader:
        images = batch["images"].to(DEVICE, non_blocking=PIN_MEMORY)
        wind_seq = batch["wind_seq"].to(DEVICE, non_blocking=PIN_MEMORY)
        wind_stats = batch["wind_stats"].to(DEVICE, non_blocking=PIN_MEMORY)
        last_wind = batch["last_wind"].to(DEVICE, non_blocking=PIN_MEMORY)
        with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
            residual = model(images, wind_seq, wind_stats)
        prediction = residual.float() + last_wind.unsqueeze(1)
        # 물리적 범위로 클리핑 — RMSE 는 큰 오차에 제곱 페널티라 극단값 방어가 효과적입니다.
        prediction = prediction.clamp(CLIP_LOW, CLIP_HIGH)
        predictions.append(prediction.cpu().numpy())
        sample_ids.extend(batch["sample_id"])
    return np.concatenate(predictions).astype(np.float64), sample_ids


validation_prediction, validation_ids = predict(val_loader)
assert validation_ids == val_inputs.sample_id.tolist()

model_score, model_per_horizon = official_rmse(val_targets, validation_prediction)
validation_metrics = metrics_by_horizon(val_targets, validation_prediction, val_persistence)
validation_metrics.to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)

print(f"공식 RMSE (mean of horizon RMSE) : {model_score:8.3f} km/s")
print(f"pooled RMSE (전체 원소)          : {pooled_rmse(val_targets, validation_prediction):8.3f} km/s")
print(f"persistence 공식 RMSE            : {persistence_score:8.3f} km/s")
print(f"persistence 대비 개선             : {persistence_score - model_score:8.3f} km/s"
      f"  ({(persistence_score - model_score) / persistence_score:.1%})")
if model_score >= persistence_score:
    print("\n>>> 경고: persistence 를 이기지 못했습니다. 제출하지 말고 학습 설정을 점검하세요.")

figure, axis = plt.subplots(figsize=(7, 4))
axis.plot(validation_metrics.horizon_h, validation_metrics.rmse, marker="o", label="model")
axis.plot(validation_metrics.horizon_h, validation_metrics.persistence_rmse,
          marker="s", linestyle="--", label="persistence")
axis.set_xlabel("forecast horizon (h)")
axis.set_ylabel("RMSE (km/s)")
axis.grid(alpha=0.3)
axis.legend()
plt.tight_layout()
plt.show()

validation_metrics

## 9. Test 추론 · submission.csv 생성

In [ ]:
del train_loader, val_loader, train_dataset, val_dataset
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

test_dataset = SolarWindDataset(
    test_image_array, test_image_index, test_inputs,
    test_wind, test_wind_valid, targets=None,
)
test_loader = make_loader(test_dataset, shuffle=False)
test_prediction, predicted_ids = predict(test_loader)

assert predicted_ids == test_inputs.sample_id.tolist()
assert test_prediction.shape == (len(test_inputs), 12)
assert np.isfinite(test_prediction).all()

submission = pd.DataFrame(test_prediction, columns=TARGET_COLUMNS)
submission.insert(0, "sample_id", predicted_ids)
submission.to_csv(SUBMISSION_DIR / "submission.csv", index=False)

# 최종 추론 모델을 제출 폴더에 복사
shutil.copyfile(checkpoint_path, SUBMISSION_DIR / "model.pth")

assert submission.columns.tolist() == ["sample_id"] + TARGET_COLUMNS
assert submission.sample_id.is_unique
print("saved:", (SUBMISSION_DIR / "submission.csv").resolve())
print("shape:", submission.shape)
print(submission[TARGET_COLUMNS].describe().loc[["mean", "std", "min", "max"]].round(1))

del test_dataset, test_loader
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

submission.head()

## 10. 제출 전 체크리스트

`submission/` 아래 **정확히 3개 파일**만 제출됩니다: `code.ipynb`, `model.pth`, `submission.csv`.

> ⚠️ `code.ipynb` 는 이 노트북 자신입니다. **저장(Ctrl+S) 후 `submission/code.ipynb` 로 복사**하세요.
> 노트북이 자기 자신을 안정적으로 복사할 방법이 없어 이 단계는 수동입니다.

In [ ]:
EXPECTED_TEST_ROWS = 3868

print("=== 제출 점검 ===")
ok = True
for name in ["code.ipynb", "model.pth", "submission.csv"]:
    path = SUBMISSION_DIR / name
    if path.exists():
        print(f"  [O] {name:16s} {path.stat().st_size / 1024 ** 2:8.2f} MiB")
    else:
        print(f"  [X] {name:16s} 없음")
        ok = False

check = pd.read_csv(SUBMISSION_DIR / "submission.csv")
print()
print(f"  행 수      : {len(check):,} (기대 {EXPECTED_TEST_ROWS:,})",
      "OK" if len(check) == EXPECTED_TEST_ROWS else "<-- 불일치")
print(f"  컬럼       : {check.columns.tolist() == ['sample_id'] + TARGET_COLUMNS}")
print(f"  결측       : {int(check[TARGET_COLUMNS].isna().sum().sum())}")
print(f"  sample_id  : 유일={check.sample_id.is_unique}, "
      f"test_ids 일치={sorted(check.sample_id) == sorted(test_ids.sample_id)}")
print(f"  값 범위    : [{check[TARGET_COLUMNS].to_numpy().min():.1f}, "
      f"{check[TARGET_COLUMNS].to_numpy().max():.1f}] km/s")

extra = [p.name for p in SUBMISSION_DIR.iterdir()
         if p.name not in {"code.ipynb", "model.pth", "submission.csv"}]
if extra:
    print(f"\n  주의: 불필요한 파일이 있습니다 -> {extra}")

print()
print("규정 확인:")
print("  - validation 을 학습에 사용하지 않음        : O (val_loader 는 평가에만 사용)")
print("  - pretrained weight 미사용                 : O (build_model 은 random init)")
print("  - test target 추정/역산 미사용              : O")
print("  - 이 노트북 하나로 실행 가능                : O (외부 import 없음)")
print()
print("남은 수동 작업: 노트북 저장 후 submission/code.ipynb 로 복사")
if not ok:
    print(">>> 위 [X] 항목을 해결한 뒤 제출하세요.")